# 通用的方式：Agent->AgentExecutor
## 1、 FUNCTION_CALL的模式

In [2]:
from langchain_openai import ChatOpenAI
import dotenv
from langchain_core.tools import Tool
import os
#1、提供工具
# 1.导入相关包
from langchain.agents import AgentExecutor, create_tool_calling_agent, create_react_agent
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_core.prompts import ChatPromptTemplate

# 2.定义搜索化工具
# ① 设置 TAVILY_API 密钥
dotenv.load_dotenv()
os.environ["TAVILY_API_KEY"] = os.getenv("TAVILY_API_KEY")  # 需要替换为你的 Tavily API 密钥

# ② 定义搜索工具
search = TavilySearchResults(max_results=1)
search_tool = Tool(
    name="search_tool",
    func=search.run,
    description="用于互联网信息的检索"
)

#2、提供工具集
tools = [search_tool]

#3、提供大模型
dotenv.load_dotenv()

os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY1")
os.environ['OPENAI_BASE_URL'] = os.getenv("OPENAI_BASE_URL")
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
)
#4、提供提示词模板 （明显相对于传统模式的区别）
prompt_template = ChatPromptTemplate.from_messages([
    ("system", "你是一个人工智能的助手，必要时根据用户的需求，即时调用TavilySearch工具进行网络搜索"),
    ("human", "我的问题是：{question}"),
    ("system", "{agent_scratchpad}")  #必须在提示词模板中提供agent_scratchpad参数。
])

#5、创建Agent
agent = create_tool_calling_agent(
    llm=llm,
    tools=tools,
    prompt=prompt_template,
)

#6、创建AgentExecutor
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,  #显示详情
)

#7、调用
agent_executor.invoke({"question": "今年9月3号北京会有阅兵吗？"})




> Entering new AgentExecutor chain...

Invoking: `search_tool` with `2023年9月3号 北京 阅兵`


[{'url': 'https://zh.wikipedia.org/wiki/%E7%BA%AA%E5%BF%B5%E4%B8%AD%E5%9B%BD%E4%BA%BA%E6%B0%91%E6%8A%97%E6%97%A5%E6%88%98%E4%BA%89%E6%9A%A8%E4%B8%96%E7%95%8C%E5%8F%8D%E6%B3%95%E8%A5%BF%E6%96%AF%E6%88%98%E4%BA%89%E8%83%9C%E5%88%A980%E5%91%A8%E5%B9%B4%E5%A4%A7%E4%BC%9A', 'content': '中央广播电视总台所属中国中央电视台综合频道和新闻频道于9月3日6:00-12:00并机直播《纪念中国人民抗日战争暨世界反法西斯战争胜利80周年特别报道》，而中文国际频道和国防军事频道则于6:00-10:40并机直播，4K超高清频道则于8:45-10:30实况转播大会现场。中国中央电视台《新闻联播》主持人宝晓峰和刚强担任现场解说。\n\n总台中国环球电视网各自分别制作阅兵报道并配以相对应的语言解说。主频道 "中国环球电视网 (频道)")播出时间为06:30-11:00，法语频道、俄语频道、阿语频道及西语频道播出时间为07:00-11:00。\n\n总台中央人民广播电台除香港之声、老年之声、阅读之声、中国乡村之声、轻松调频及劲曲调频 "Hit FM (中国国际广播电台)")外，旗下广播电台同步以普通话解说于7:30-10:40转播大会。\n\n总台中国国际广播电台除环球资讯广播外，还使用了汉语普通话、英语、俄语、日语、德语等对外进行现场直播。 [...] 上午临近9时，现场倒计时，大屏幕显示1945、1955、1965……2025数字。上午9时，中共中央政治局常委、国务院总理李强宣布纪念中国人民抗日战争暨世界反法西斯战争胜利80周年大会开始。由中国人民解放军仪仗大队礼炮分队于正阳门分两外八列镇鸣80响礼炮，象征中国人民抗日战争暨世界反法西斯战争胜利80年。 中国人民解放军仪仗大队国旗护卫伴随礼炮轰鸣，护卫国旗从人民英雄纪念碑下出发，穿过写有

{'question': '今年9月3号北京会有阅兵吗？',
 'output': '今年9月3日，北京将举行纪念中国人民抗日战争暨世界反法西斯战争胜利80周年的活动，其中包括阅兵式。活动将于上午9时开始，届时会有相关的直播和报道。'}

## 2、 React的模式

In [3]:
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
import dotenv
from langchain_core.tools import Tool
import os
#1、提供工具
# 1.导入相关包
from langchain.agents import AgentExecutor
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_core.prompts import ChatPromptTemplate
from langchain.agents.react.agent import create_react_agent

# 2.定义搜索化工具
# ① 设置 TAVILY_API 密钥
dotenv.load_dotenv()
os.environ["TAVILY_API_KEY"] = os.getenv("TAVILY_API_KEY")  # 需要替换为你的 Tavily API 密钥

# ② 定义搜索工具
search = TavilySearchResults(max_results=1)
search_tool = Tool(
    name="search_tool",
    func=search.run,
    description="用于互联网信息的检索"
)

#2、提供工具集
tools = [search_tool]

#3、提供大模型
dotenv.load_dotenv()

os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY1")
os.environ['OPENAI_BASE_URL'] = os.getenv("OPENAI_BASE_URL")
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
)
#4、提供提示词模板 

#  使用PromptTemplate:
template = '''Answer the following questions as best you can. You have access to the following tools:
            {tools}

            Use the following format:

            Question: the input question you must answer
            Thought: you should always think about what to do
            Action: the action to take, should be one of [{tool_names}]
            Action Input: the input to the action
            Observation: the result of the action
            ... (this Thought/Action/Action Input/Observation can repeat N times)
            Thought: I now know the final answer
            Final Answer: the final answer to the original input question

            Begin!

            Question: {input}
            Thought:{agent_scratchpad}'''
prompt_template = PromptTemplate(
    template=template,
)

#上面代码使用的是create_tool_calling_agent，这里使用的是create_react_agent
agent = create_react_agent(
    llm=llm,
    tools=tools,
    prompt=prompt_template,
)

#6、创建AgentExecutor
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,  #显示详情
    handle_parsing_errors=True
)

#7、调用
agent_executor.invoke({"input": "明天北京的天气情况如何？"})




> Entering new AgentExecutor chain...
我需要查找明天北京的天气预报。  
Action: search_tool  
Action Input: "明天 北京 天气预报"  [{'url': 'https://weather.cma.cn/web/weather/', 'content': '|  |  |  |  |  |  |  |  |  |\n ---  ---  ---  --- \n| 时间 | 11:00 | 14:00 | 17:00 | 20:00 | 23:00 | 02:00 | 05:00 | 08:00 |\n| 天气 |  |  |  |  |  |  |  |  |\n| 气温 | 24.2℃ | 28.3℃ | 28.8℃ | 26.1℃ | 23.8℃ | 23.4℃ | 22.2℃ | 25.3℃ |\n| 降水 | 2.3mm | 2.3mm | 2.3mm | 2.3mm | 0.9mm | 无降水 | 0.7mm | 无降水 |\n| 风速 | 3.1m/s | 3.3m/s | 3.3m/s | 2.6m/s | 2.8m/s | 3m/s | 3.3m/s | 3.3m/s |\n| 风向 | 东北风 | 西南风 | 东南风 | 西南风 | 西南风 | 东南风 | 东北风 | 东南风 | [...] |  |  |  |  |  |  |  |  |  |\n ---  ---  ---  --- \n| 时间 | 08:00 | 11:00 | 14:00 | 17:00 | 20:00 | 23:00 | 02:00 | 05:00 |\n| 天气 |  |  |  |  |  |  |  |  |\n| 气温 | 25.9℃ | 27.7℃ | 30.8℃ | 29.4℃ | 27.4℃ | 26℃ | 24.2℃ | 25.4℃ |\n| 降水 | 无降水 | 2.3mm | 2.3mm | 2.3mm | 2.3mm | 无降水 | 无降水 | 1.9mm |\n| 风速 | 3.2m/s | 3.2m/s | 3.3m/s | 2.5m/s | 2.6m/s | 2.6m/s | 2.4m/s | 2.8m/s |\n| 风向 | 西南风 | 西南风 | 西南风 | 

{'input': '明天北京的天气情况如何？', 'output': '明天北京的天气情况是：多云，有小雨，气温在24.2℃到30.8℃之间，风速较轻。'}